# Ingesta Streaming - Farmia Lakehouse

Pipeline de ingesta en tiempo real para el proyecto **Farmia**. Consume mensajes de Confluent Cloud (Kafka), los persiste en la capa **Bronze** de Azure Data Lake Storage y valida los resultados. Se procesan dos flujos: `eventos_clientes` y `sensores_iot`.

## 1. Reinicio del entorno Python
Se reinicia el intérprete para garantizar un estado limpio antes de ejecutar el pipeline de streaming.

In [0]:
%restart_python

## 2. Importación de módulos
Se añade el directorio raíz del proyecto al `sys.path`, se configura el entorno como `databricks` y se importan el motor de ingesta streaming (`kafka_engine`) y el script de auditoría.

In [0]:
import sys
import os

sys.path.append(os.path.abspath(".."))
os.environ["ENVIRONMENT"] = "databricks"

from src.streaming.kafka_engine import run_streaming_ingestion
from scripts.query_audit import show_latest_audit

## 3. Generación de datos sintéticos
Se ejecuta el generador que simula las fuentes de datos del negocio Farmia y publica mensajes en los topics de Confluent Cloud (Kafka) para su posterior consumo en streaming.

In [0]:
import scripts.generate_synthetic_data as gen

gen.main(mode="streaming")

## 4. Ingesta streaming — `eventos_clientes`
Consumo del topic `farmia.events.customers` de Confluent Cloud. Con `availableNow=True` se procesa el backlog completo, se persiste en formato Delta en la capa **Bronze** de ADLS y la celda finaliza automáticamente.

In [0]:
# Con availableNow=True, consumirá el backlog de Confluent Cloud,
# guardará en ADLS Bronze y la celda finalizará sola.
run_streaming_ingestion("eventos_clientes.json")

## 5. Auditoría — `eventos_clientes`
Se consulta la tabla de control de ingesta para verificar la trazabilidad y el resultado del flujo de `eventos_clientes`.

In [0]:
# Comprobar el resultado en Delta Lake
show_latest_audit()

## 6. Validación — `eventos_clientes`
Consulta directa sobre la tabla Delta en Bronze para verificar que los registros de eventos de clientes se han persistido correctamente.

In [0]:
%sql
SELECT * 
FROM delta.`${spark.farmia.base_bronze_path}/eventos_clientes` 
LIMIT 10;

## 7. Ingesta streaming — `sensores_iot`
Consumo del topic `farmia.iot.sensors.*` de Confluent Cloud. Se ingestan las lecturas de sensores IoT (temperatura, humedad del suelo, pH) en formato Delta en la capa **Bronze** de ADLS.

In [0]:
# Ingesta streaming de sensores IoT desde Confluent Cloud
run_streaming_ingestion("sensores_iot.json")

## 8. Auditoría — `sensores_iot`
Se consulta la tabla de control de ingesta para verificar la trazabilidad y el resultado del flujo de `sensores_iot`.

In [0]:
# Comprobar el resultado en Delta Lake
show_latest_audit()

## 9. Validación — `sensores_iot`
Consulta directa sobre la tabla Delta en Bronze para verificar que las lecturas de sensores IoT se han persistido correctamente.

In [0]:
%sql
SELECT * 
FROM delta.`${spark.farmia.base_bronze_path}/sensores_iot` 
LIMIT 10;